In [0]:
from pyspark.sql import functions as F

login = "mariiasirska"
catalog = "dbr_dev_ua5816bd"
silver = f"{catalog}.{login}_silver"

def st(name):
    return f"{silver}.olist_{name}_silver"

QUAR = f"{silver}.olist_quarantine"

Delta CHECK constraints:

In [0]:
constraints = [
    ("orders", "chk_order_status",
     "order_status IN ('created','approved','invoiced','processing','shipped','delivered','unavailable','canceled')"),
    ("order_items", "chk_price_non_negative", "price >= 0"),
    ("order_items", "chk_freight_non_negative", "freight_value >= 0"),
    ("reviews", "chk_review_score", "review_score BETWEEN 1 AND 5"),
]

In [0]:
for table, name, expr in constraints:
    try:
        spark.sql(f"ALTER TABLE {st(table)} ADD CONSTRAINT {name} CHECK ({expr})")
        print(f"added {name} on {table}")
    except Exception as e:
        print(f"{name} on {table}: {str(e)[:150]}")


In [0]:
row = spark.table(st("order_items")).select("order_id", "order_item_id").first()
try:
    spark.sql(f"""
    UPDATE {st('order_items')} SET price = -1
    WHERE order_id = '{row['order_id']}' AND order_item_id = {row['order_item_id']}
    """)
except Exception as e:
    print("REJECTED by CHECK constraint:", type(e).__name__)
    print(str(e)[:300])

Quarantine summary:![](path)

In [0]:

display(spark.table(QUAR).groupBy("table_name", "reason").count().orderBy("table_name"))

Data quality checks on silver:

In [0]:

checks = [
    ("orders", ["order_id"]),
    ("customers", ["customer_id"]),
    ("products", ["product_id"]),
    ("order_items", ["order_id", "order_item_id"]),
    ("reviews", ["review_id", "order_id"]),
]

rows = []
for table, keys in checks:
    dups = spark.table(st(table)).groupBy(*keys).count().filter("count > 1").count()
    rows.append((table, "unique key: " + ", ".join(keys), dups, "PASS" if dups == 0 else "FAIL"))


In [0]:
bad = (
    spark.table(st("sellers")).groupBy("seller_id")
    .agg(F.sum(F.col("is_current").cast("int")).alias("n_current"))
    .filter("n_current <> 1").count()
)
rows.append(("sellers", "exactly one current row per seller_id", bad, "PASS" if bad == 0 else "FAIL"))

display(spark.createDataFrame(rows, "table string, check string, violations long, status string"))

Table layout:

In [0]:
src = st("orders")
variants = {
    "partitioned": (
        f"CREATE OR REPLACE TABLE {silver}.olist_orders_part PARTITIONED BY (order_status) AS SELECT * FROM {src}",
        None,
    ),
    "zorder": (
        f"CREATE OR REPLACE TABLE {silver}.olist_orders_zorder AS SELECT * FROM {src}",
        f"OPTIMIZE {silver}.olist_orders_zorder ZORDER BY (customer_id)",
    ),
    "liquid": (
        f"CREATE OR REPLACE TABLE {silver}.olist_orders_liquid CLUSTER BY (customer_id, purchase_ts) AS SELECT * FROM {src}",
        f"OPTIMIZE {silver}.olist_orders_liquid",
    ),
}
tables = {"partitioned": "olist_orders_part", "zorder": "olist_orders_zorder", "liquid": "olist_orders_liquid"}

for name, (create_sql, optimize_sql) in variants.items():
    spark.sql(create_sql)
    if optimize_sql:
        spark.sql(optimize_sql)

wanted = ["name", "numFiles", "sizeInBytes", "partitionColumns", "clusteringColumns"]
for name, tbl in tables.items():
    d = spark.sql(f"DESCRIBE DETAIL {silver}.{tbl}")
    display(d.select([c for c in wanted if c in d.columns]))


In [0]:
for name, tbl in tables.items():
    spark.sql(f"SELECT * FROM {silver}.{tbl} WHERE customer_id = (SELECT min(customer_id) FROM {src})").collect()
    print(name, "ok")

In [0]:
spark.sql(f"ALTER TABLE {st('orders')} CLUSTER BY (customer_id, purchase_ts)")
spark.sql(f"OPTIMIZE {st('orders')}")
display(spark.sql(f"DESCRIBE DETAIL {st('orders')}"))

In [0]:
for tbl in tables.values():
    spark.sql(f"DROP TABLE IF EXISTS {silver}.{tbl}")


 Scheduled job:

In [0]:

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

cluster_id = "0828-065013-gln8ogbm"
nb_dir = "/Workspace/Users/sirskam123@softserve.academy/databricks_labs/lab4"
job_name = f"{login}lab4_silver_pipeline"

w = WorkspaceClient()

existing = list(w.jobs.list(name=job_name))
if existing:
    print("job already exists:", existing[0].job_id)
else:
    job = w.jobs.create(
        name=job_name,
        tasks=[
            jobs.Task(
                task_key="silver_pipeline",
                existing_cluster_id=cluster_id,
                notebook_task=jobs.NotebookTask(notebook_path=f"{nb_dir}/2_silver_pipeline"),
            ),
            jobs.Task(
                task_key="maintenance",
                depends_on=[jobs.TaskDependency(task_key="silver_pipeline")],
                existing_cluster_id=cluster_id,
                notebook_task=jobs.NotebookTask(notebook_path=f"{nb_dir}/5_maintenance"),
            ),
        ],
        schedule=jobs.CronSchedule(
            quartz_cron_expression="0 0 3 * * ?",
            timezone_id="UTC",
            pause_status=jobs.PauseStatus.PAUSED,
        ),
        max_concurrent_runs=1,
    )
    print("job_id:", job.job_id)

In [0]:
job_id = existing[0].job_id if existing else job.job_id
print(w.jobs.run_now(job_id=job_id).result().state.result_state)